# 🔬 CNN2D with a real ECG input (Dataset B)

**Question:** does adding the ECG mean-power feature change seizure detection when it is fed to the network as a genuine second input?

- `eeg_only`: CNN2D on the 90 EEG band-power features (control)
- `eeg_ecg`: the same CNN2D plus a small Dense branch for `ECG_PROXY_mean_power`, concatenated before the 128-D feature vector (so the SVM embedding also contains ECG)

Both conditions use the **same split and seed**, so differences are paired. ECG is `log10`-transformed, then z-scored with training statistics only. Early stopping uses an **exact** validation AUC (Keras' built-in AUC is binned and reads 0.5 when all scores are tiny). The SVM keeps all positive training windows and subsamples only negatives (10 per positive).

**Caveats:** random window-level split (Dataset B has only 2 patients with seizures, so a patient-wise split is not possible); about 16 positive test windows per seed, so judge the paired difference over seeds, not a single run.

## 0. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(g, True)
print('GPU:', tf.config.list_physical_devices('GPU') or 'none (will be slow)')

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import os, gc, time, numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, average_precision_score)
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models, Input, regularizers, optimizers as opt_module

BAND_NAMES = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
ECG_COL = "ECG_PROXY_mean_power"

In [4]:
# ── Data ─────────────────────────────────────────────────────────────────────
def load_dataset_b(path):
    df = pd.read_csv(path)
    # drop corrupted CSV lines (their "patient" is a number fragment, not chbNN)
    df = df[df["patient"].astype(str).str.fullmatch(r"chb\d\d")]
    df = df.dropna().reset_index(drop=True)
    df["label"] = df["label"].astype(int)
    return df


def get_band_columns(df):
    return {b: [c for c in df.columns if c.startswith(f"{b}_")] for b in BAND_NAMES}


def prepare_inputs(df):
    """EEG tensor (N, n_ch, 5, 1), raw ECG mean power (float64), labels."""
    bc = get_band_columns(df)
    X = np.stack([df[bc[b]].values for b in BAND_NAMES], axis=1)   # (N, 5, n_ch)
    X = X[..., np.newaxis].transpose(0, 2, 1, 3).astype(np.float32)  # (N, n_ch, 5, 1)
    ecg = df[ECG_COL].values.astype(np.float64)   # keep float64: values are ~1e-8
    y = df["label"].values.astype(int)
    return X, ecg, y

In [5]:
# ── Model: same CNN2D as notebook 03, plus an optional real ECG input ────────
def build_cnn2d_ecg(n_channels=18, n_bands=5, use_ecg=False, feature_dim=128,
                    dropout=0.4, l2=1e-4, output_bias=None):
    bias_init = (tf.keras.initializers.Constant(output_bias)
                 if output_bias is not None else "zeros")
    eeg_in = Input(shape=(n_channels, n_bands, 1), name="eeg_input")
    x = eeg_in
    for filters, pool, drop in [(32, True, dropout / 2), (64, True, dropout / 2),
                                (128, False, dropout)]:
        x = layers.Conv2D(filters, (1, 3), padding="same", activation="relu",
                          kernel_regularizer=regularizers.l2(l2))(x)
        x = layers.BatchNormalization()(x)
        if pool:
            x = layers.MaxPooling2D((2, 1), padding="same")(x)
        x = layers.SpatialDropout2D(drop)(x)
    x = layers.GlobalAveragePooling2D()(x)

    inputs = [eeg_in]
    if use_ecg:
        ecg_in = Input(shape=(1,), name="ecg_input")
        # small init: the ECG branch starts nearly silent and only gains influence if it helps
        # (a full-scale noisy input slows early EEG learning and would bias the comparison)
        e = layers.Dense(16, activation="relu", name="ecg_dense",
                         kernel_initializer=tf.keras.initializers.RandomNormal(stddev=0.01))(ecg_in)
        x = layers.Concatenate(name="eeg_ecg_concat")([x, e])
        inputs.append(ecg_in)

    feat = layers.Dense(feature_dim, activation="relu", name="feature_vector",
                        kernel_regularizer=regularizers.l2(l2))(x)
    feat = layers.Dropout(dropout)(feat)
    out = layers.Dense(1, activation="sigmoid", name="output",
                       bias_initializer=bias_init)(feat)
    full = models.Model(inputs, out, name="CNN2D_EEG_ECG" if use_ecg else "CNN2D_EEG")
    feat_m = models.Model(inputs, feat, name="feat")
    return full, feat_m



class ExactValAUC(tf.keras.callbacks.Callback):
    """Exact (rank-based) validation AUC each epoch. Keras' streaming AUC uses 200
    fixed thresholds, so it reads exactly 0.5 whenever all predictions are tiny."""
    def __init__(self, val_in, yv):
        super().__init__()
        self.val_in, self.yv = val_in, yv
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.val_in, batch_size=1024, verbose=0).ravel()
        logs["val_auc_exact"] = float(roc_auc_score(self.yv, p))

def compile_model(m, opt):
    m.compile(optimizer=opt, loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m


def make_opt(name):
    if name == "Adam":
        return opt_module.Adam(1e-3, clipnorm=1.0)
    if name == "Nadam":
        return opt_module.Nadam(1e-3, clipnorm=1.0)
    if name == "AdamW":
        return opt_module.AdamW(1e-3, weight_decay=1e-4, clipnorm=1.0)
    raise ValueError(name)

In [6]:
# ── SVM on the embedding: keep ALL positives, subsample only negatives ───────
def evaluate_with_svm(Ftr, ytr, Fte, yte, neg_per_pos=10, seed=42):
    pos = np.where(ytr == 1)[0]
    neg = np.where(ytr == 0)[0]
    rng = np.random.RandomState(seed)
    neg_s = rng.choice(neg, min(len(neg), neg_per_pos * len(pos)), replace=False)
    idx = np.concatenate([pos, neg_s])
    sc = StandardScaler()
    Xtr = sc.fit_transform(Ftr[idx])
    Xte = sc.transform(Fte)
    svm = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42)
    svm.fit(Xtr, ytr[idx])
    p = svm.predict(Xte)
    tn, fp, fn, tp = confusion_matrix(yte, p, labels=[0, 1]).ravel()
    return {
        "svm_acc": round(accuracy_score(yte, p), 4),
        "svm_sens": float(round(tp / (tp + fn), 4)) if (tp + fn) else 0.0,
        "svm_spec": float(round(tn / (tn + fp), 4)) if (tn + fp) else 0.0,
        "svm_f1_macro": round(f1_score(yte, p, average="macro"), 4),
    }

In [7]:
# ── One trial ────────────────────────────────────────────────────────────────
def run_trial(X_eeg, ecg_raw, y, condition, seed, opt_name, epochs=50,
              batch_size=64, patience=8):
    use_ecg = (condition == "eeg_ecg")
    tf.keras.backend.clear_session()
    tf.random.set_seed(seed)
    np.random.seed(seed)

    # same split for both conditions (depends only on seed)
    idx = np.arange(len(y))
    i_trv, i_te = train_test_split(idx, test_size=0.15, stratify=y, random_state=seed)
    i_tr, i_va = train_test_split(i_trv, test_size=0.15 / 0.85,
                                  stratify=y[i_trv], random_state=seed)

    # normalise with TRAIN statistics only
    mu = X_eeg[i_tr].mean(axis=0, keepdims=True)
    sd = X_eeg[i_tr].std(axis=0, keepdims=True)
    f = lambda i: ((X_eeg[i] - mu) / (sd + 1e-8)).astype(np.float32)
    Xtr, Xv, Xte = f(i_tr), f(i_va), f(i_te)

    ecg_log = np.log10(ecg_raw + 1e-18)          # values are ~1e-8: log10, not log1p
    e_mu, e_sd = ecg_log[i_tr].mean(), ecg_log[i_tr].std()
    g = lambda i: ((ecg_log[i] - e_mu) / (e_sd + 1e-8)).astype(np.float32).reshape(-1, 1)
    Etr, Ev, Ete = g(i_tr), g(i_va), g(i_te)
    ytr, yv, yte = y[i_tr], y[i_va], y[i_te]

    # ECG alone: direction chosen on TRAIN only, scored on test (0.5 = no information)
    a_tr = roc_auc_score(ytr, Etr.ravel())
    sign = 1.0 if a_tr >= 0.5 else -1.0
    ecg_alone_auc = roc_auc_score(yte, sign * Ete.ravel())

    total, n1 = len(ytr), int(ytr.sum())
    cw = {0: total / (2 * (total - n1)), 1: total / (2 * n1)}
    output_bias = float(np.log(n1 / (total - n1)))

    full_m, feat_m = build_cnn2d_ecg(n_channels=X_eeg.shape[1], n_bands=X_eeg.shape[2],
                                     use_ecg=use_ecg, output_bias=output_bias)
    full_m = compile_model(full_m, make_opt(opt_name))
    tr_in = [Xtr, Etr] if use_ecg else Xtr
    va_in = [Xv, Ev] if use_ecg else Xv
    te_in = [Xte, Ete] if use_ecg else Xte
    print(f"    model inputs: {[i.shape for i in full_m.inputs]}")

    cbs = [ExactValAUC(va_in, yv),   # must come before EarlyStopping
           tf.keras.callbacks.EarlyStopping(monitor="val_auc_exact", mode="max", patience=patience,
                                            restore_best_weights=True, verbose=0),
           tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4,
                                                min_lr=1e-6, verbose=0)]
    t0 = time.time()
    hist = full_m.fit(tr_in, ytr, validation_data=(va_in, yv), epochs=epochs,
                      batch_size=batch_size, class_weight=cw, callbacks=cbs, verbose=0)
    elapsed = round((time.time() - t0) / 60, 2)

    p_te = full_m.predict(te_in, batch_size=1024, verbose=0).ravel()
    row = {
        "condition": condition, "seed": seed, "optimizer": opt_name,
        "n_train_pos": n1, "n_val_pos": int(yv.sum()), "n_test_pos": int(yte.sum()),
        "best_val_auc": round(max(hist.history["val_auc_exact"]), 4),
        "test_auc": round(roc_auc_score(yte, p_te), 4),
        "test_auprc": round(average_precision_score(yte, p_te), 4),
        "ecg_alone_auc": round(ecg_alone_auc, 4),
        "epochs_run": len(hist.history["val_auc_exact"]), "train_time_min": elapsed,
    }
    f_tr = feat_m.predict(tr_in, batch_size=1024, verbose=0)
    f_te = feat_m.predict(te_in, batch_size=1024, verbose=0)
    row.update(evaluate_with_svm(f_tr, ytr, f_te, yte))
    del full_m, feat_m
    gc.collect()
    tf.keras.backend.clear_session()
    return row


def run_all(df, results_path, seeds, optimizers, conditions=("eeg_only", "eeg_ecg"),
            epochs=50):
    X_eeg, ecg_raw, y = prepare_inputs(df)
    done = set()
    if os.path.exists(results_path):
        prev = pd.read_csv(results_path)
        done = set(zip(prev["condition"], prev["seed"], prev["optimizer"]))
        print(f"[resume] {len(done)} trial(s) already saved, skipping those")
    for seed in seeds:
        for opt_name in optimizers:
            for cond in conditions:
                if (cond, seed, opt_name) in done:
                    continue
                print(f"\n=== {cond} | seed {seed} | {opt_name} ===")
                row = run_trial(X_eeg, ecg_raw, y, cond, seed, opt_name, epochs=epochs)
                print("   ", row)
                os.makedirs(os.path.dirname(results_path), exist_ok=True)
                pd.DataFrame([row]).to_csv(results_path, mode="a", index=False,
                                           header=not os.path.exists(results_path))


def summarize(results_path):
    r = pd.read_csv(results_path)
    cols = ["test_auc", "test_auprc", "best_val_auc", "svm_sens", "svm_spec", "ecg_alone_auc"]
    print("=== mean ± std over seeds, per condition ===")
    for cond, g in r.groupby("condition"):
        print(f"\n{cond}  (n={len(g)}, test positives per seed: {g['n_test_pos'].tolist()})")
        for c in cols:
            print(f"  {c:14s} {g[c].mean():.4f} ± {g[c].std(ddof=0):.4f}")
    w = r.pivot_table(index=["seed", "optimizer"], columns="condition", values="test_auc")
    if {"eeg_only", "eeg_ecg"} <= set(w.columns):
        d = (w["eeg_ecg"] - w["eeg_only"]).dropna()
        print("\n=== paired test-AUC difference (eeg_ecg - eeg_only), same split per seed ===")
        print(d.round(4).to_string())
        print(f"mean {d.mean():+.4f}  std {d.std(ddof=0):.4f}  "
              f"(ECG helped in {(d > 0).sum()} of {len(d)} pairs)")

## Run
Edit `SEEDS` / `OPTIMIZERS` if needed. Finished trials are skipped automatically if the runtime disconnects.

In [8]:
DATASET_B   = "/content/drive/MyDrive/epilepsy_outputs/dataset_B.csv"
RESULTS_ECG = "/content/drive/MyDrive/epilepsy_outputs/results_ecg_cnn2d.csv"
SEEDS       = [42, 43, 44]
OPTIMIZERS  = ["Adam"]      # add "Nadam", "AdamW" for more
EPOCHS      = 50

dfB = load_dataset_b(DATASET_B)
print("rows:", len(dfB), "| patients:", dfB.patient.nunique(), "| records:", dfB.groupby(['patient','record']).ngroups)
print("positives per patient:", dfB[dfB.label==1].groupby('patient').size().to_dict())
print(dfB[ECG_COL].describe())

rows: 164386 | patients: 3 | records: 39
positives per patient: {'chb04': 74, 'chb12': 30}
count    1.643860e+05
mean     1.013298e-08
std      1.540863e-08
min      2.355284e-17
25%      2.450299e-09
50%      5.511235e-09
75%      1.130320e-08
max      3.991714e-07
Name: ECG_PROXY_mean_power, dtype: float64


In [9]:
run_all(dfB, RESULTS_ECG, SEEDS, OPTIMIZERS, epochs=EPOCHS)


=== eeg_only | seed 42 | Adam ===
    model inputs: [(None, 18, 5, 1)]
    {'condition': 'eeg_only', 'seed': 42, 'optimizer': 'Adam', 'n_train_pos': 72, 'n_val_pos': 16, 'n_test_pos': 16, 'best_val_auc': 0.9447, 'test_auc': np.float64(0.9194), 'test_auprc': np.float64(0.0901), 'ecg_alone_auc': np.float64(0.6864), 'epochs_run': 10, 'train_time_min': 1.72, 'svm_acc': 0.9491, 'svm_sens': 0.9375, 'svm_spec': 0.9492, 'svm_f1_macro': 0.4986}

=== eeg_ecg | seed 42 | Adam ===
    model inputs: [(None, 18, 5, 1), (None, 1)]
    {'condition': 'eeg_ecg', 'seed': 42, 'optimizer': 'Adam', 'n_train_pos': 72, 'n_val_pos': 16, 'n_test_pos': 16, 'best_val_auc': 0.9874, 'test_auc': np.float64(0.9252), 'test_auprc': np.float64(0.1183), 'ecg_alone_auc': np.float64(0.6864), 'epochs_run': 27, 'train_time_min': 3.92, 'svm_acc': 0.9568, 'svm_sens': 0.9375, 'svm_spec': 0.9569, 'svm_f1_macro': 0.5027}

=== eeg_only | seed 43 | Adam ===
    model inputs: [(None, 18, 5, 1)]
    {'condition': 'eeg_only', 'seed':

## Summary
This is the only output I need you to send back.

In [10]:
summarize(RESULTS_ECG)

=== mean ± std over seeds, per condition ===

eeg_ecg  (n=3, test positives per seed: [16, 16, 16])
  test_auc       0.9517 ± 0.0194
  test_auprc     0.1761 ± 0.0409
  best_val_auc   0.9625 ± 0.0221
  svm_sens       0.9375 ± 0.0000
  svm_spec       0.9497 ± 0.0175
  ecg_alone_auc  0.6425 ± 0.0717

eeg_only  (n=3, test positives per seed: [16, 16, 16])
  test_auc       0.9454 ± 0.0304
  test_auprc     0.2377 ± 0.1166
  best_val_auc   0.9400 ± 0.0280
  svm_sens       0.9375 ± 0.0000
  svm_spec       0.9307 ± 0.0193
  ecg_alone_auc  0.6425 ± 0.0717

=== paired test-AUC difference (eeg_ecg - eeg_only), same split per seed ===
seed  optimizer
42    Adam         0.0058
43    Adam        -0.0170
44    Adam         0.0302
mean +0.0063  std 0.0193  (ECG helped in 2 of 3 pairs)
